# Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install Detectron2

In [2]:
# Fix numpy compatibility
!pip install -q --force-reinstall numpy==1.26.4

# Build dependencies
!pip install -q setuptools==68.0.0 wheel cython

#Import detectron from the source
%cd /content

import os

if not os.path.exists("/content/detectron2"):
    !git clone https://github.com/facebookresearch/detectron2.git

%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which i

In [ ]:
import os
os.kill(os.getpid(), 9)

# Import

In [3]:
from pathlib import Path
import os, json, time
import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.model_zoo import get_config_file

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
CUDA available: True


# Path

In [4]:
SOURCE_DATASET = "DAWN"
SEED = 42
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")
CHECKPOINT_PATH = Path("/content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed42/model_final.pth")

BDD_ROOT = PROJECT_ROOT / "Datasets/processed/BDD100K"
BDD_IMAGES = BDD_ROOT / "yolo/images/val"
BDD_COCO_JSON = BDD_ROOT / "coco/annotations/instances_val.json"

RAW_RUN_DIR = PROJECT_ROOT / "Runs/faster_rcnn/dawn_bddk_seed42"
RESULTS_DIR = PROJECT_ROOT / "Results/bddk_crossdataset/Faster_rcnn/DAWN_BDDK"
RAW_RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

print("Source dataset:", SOURCE_DATASET)
print("Checkpoint:", CHECKPOINT_PATH)
print("BDD images:", BDD_IMAGES)
print("BDD annotations:", BDD_COCO_JSON)

Source dataset: DAWN
Checkpoint: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed42/model_final.pth
BDD images: /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/yolo/images/val
BDD annotations: /content/drive/MyDrive/Dissertation/Datasets/processed/BDD100K/coco/annotations/instances_val.json


# Verify paths and category order

In [5]:
assert SOURCE_DATASET in {"DAWN", "ACDC"}
assert BDD_IMAGES.exists(), f"Missing BDD image directory: {BDD_IMAGES}"
assert BDD_COCO_JSON.exists(), f"Missing annotation file: {BDD_COCO_JSON}"
assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"

with open(BDD_COCO_JSON, "r") as f:
    bdd_data = json.load(f)

bdd_categories = sorted(bdd_data.get("categories", []), key=lambda x: x["id"])
bdd_class_names = [x["name"] for x in bdd_categories]
print("Images:", len(bdd_data.get("images", [])))
print("Annotations:", len(bdd_data.get("annotations", [])))
print("Categories:", bdd_class_names)

assert bdd_class_names == CLASS_NAMES, f"Category order mismatch: {bdd_class_names} vs {CLASS_NAMES}"

Images: 1504
Annotations: 17733
Categories: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


# Register BDD100K

In [6]:
DATASET_NAME = f"bdd100k_val_{SOURCE_DATASET.lower()}_seed_{SEED}"
if DATASET_NAME in DatasetCatalog.list():
    DatasetCatalog.remove(DATASET_NAME)
    MetadataCatalog.remove(DATASET_NAME)

register_coco_instances(DATASET_NAME, {}, str(BDD_COCO_JSON), str(BDD_IMAGES))
metadata = MetadataCatalog.get(DATASET_NAME)
dataset_records = DatasetCatalog.get(DATASET_NAME)
print("Registered:", DATASET_NAME)
print("Classes:", metadata.thing_classes)
print("Images:", len(dataset_records))

Registered: bdd100k_val_dawn_seed_42
Classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
Images: 1504


# Detectron2 configuration

In [7]:
cfg = get_cfg()
cfg.merge_from_file(get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = str(CHECKPOINT_PATH)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
cfg.DATASETS.TRAIN = ()
cfg.DATASETS.TEST = (DATASET_NAME,)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.001
cfg.OUTPUT_DIR = str(RAW_RUN_DIR)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print("Device:", cfg.MODEL.DEVICE)
print("Weights:", cfg.MODEL.WEIGHTS)

Device: cuda
Weights: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed42/model_final.pth


# COCO evaluation

In [8]:
predictor = DefaultPredictor(cfg)
evaluator = COCOEvaluator(DATASET_NAME, output_dir=str(RAW_RUN_DIR))
test_loader = build_detection_test_loader(cfg, DATASET_NAME)
evaluation_results = inference_on_dataset(predictor.model, test_loader, evaluator)
evaluation_results

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0718 16:39:47.490000 8883 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Loading and preparing results...
DONE (t=0.30s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.143
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.283
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.132
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.035
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.127
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.319
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.146
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.272
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.281
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.094
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.264
 Average Recall     (AR) @[ IoU=0.50:0.

OrderedDict([('bbox',
              {'AP': 14.307605011476637,
               'AP50': 28.280709873061504,
               'AP75': 13.171307470936956,
               'APs': 3.4830421664230458,
               'APm': 12.712872126956354,
               'APl': 31.89095554535691,
               'AP-person': 16.675387944117627,
               'AP-bicycle': 7.337820535833943,
               'AP-car': 28.37417695163743,
               'AP-motorcycle': 1.6271388865915493,
               'AP-bus': 16.311951607001802,
               'AP-truck': 15.519154143677467})])

# Measure inference speed

In [9]:
for record in dataset_records[:min(10, len(dataset_records))]:
    image = cv2.imread(record["file_name"])
    _ = predictor(image)
if torch.cuda.is_available():
    torch.cuda.synchronize()

timings = []
for record in tqdm(dataset_records, desc="Measuring inference speed"):
    image = cv2.imread(record["file_name"])
    if image is None:
        raise FileNotFoundError(record["file_name"])
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    _ = predictor(image)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    timings.append(time.perf_counter() - start)

mean_s = float(np.mean(timings))
median_s = float(np.median(timings))
fps = 1.0 / mean_s if mean_s > 0 else np.nan
print("Mean inference ms:", mean_s*1000)
print("Median inference ms:", median_s*1000)
print("FPS:", fps)

Measuring inference speed:   0%|          | 0/1504 [00:00<?, ?it/s]

Mean inference ms: 52.72067388297996
Median inference ms: 52.575821000118594
FPS: 18.96789108234131


# Extract and save metrics

In [10]:
bbox = evaluation_results.get("bbox", {})
def pct(v):
    if v is None:
        return np.nan
    v = float(v)
    return np.nan if np.isnan(v) else v/100.0

global_metrics = {
    "model": "Faster R-CNN",
    "framework": "Detectron2",
    "architecture": "faster_rcnn_R_50_FPN_3x",
    "source_dataset": SOURCE_DATASET,
    "target_dataset": "BDD100K",
    "split": "val_as_external_test",
    "seed": SEED,
    "images_evaluated": len(dataset_records),
    "mAP50_95": pct(bbox.get("AP")),
    "mAP50": pct(bbox.get("AP50")),
    "mAP75": pct(bbox.get("AP75")),
    "mAP_small": pct(bbox.get("APs")),
    "mAP_medium": pct(bbox.get("APm")),
    "mAP_large": pct(bbox.get("APl")),
    "mean_inference_ms": mean_s*1000,
    "median_inference_ms": median_s*1000,
    "FPS": fps,
}

global_df = pd.DataFrame([global_metrics])
per_class_df = pd.DataFrame([
    {"class": name, "AP50_95": pct(bbox.get(f"AP-{name}"))}
    for name in CLASS_NAMES
])

global_df.to_csv(RESULTS_DIR / "global_metrics.csv", index=False)
with open(RESULTS_DIR / "global_metrics.json", "w") as f:
    json.dump(global_metrics, f, indent=2)
per_class_df.to_csv(RESULTS_DIR / "per_class_metrics.csv", index=False)
with open(RESULTS_DIR / "per_class_metrics.json", "w") as f:
    json.dump(per_class_df.replace({np.nan: None}).to_dict(orient="records"), f, indent=2)

print(global_df)
print(per_class_df)
print("Saved to:", RESULTS_DIR)

          model   framework             architecture source_dataset  \
0  Faster R-CNN  Detectron2  faster_rcnn_R_50_FPN_3x           DAWN   

  target_dataset                 split  seed  images_evaluated  mAP50_95  \
0        BDD100K  val_as_external_test    42              1504  0.143076   

      mAP50     mAP75  mAP_small  mAP_medium  mAP_large  mean_inference_ms  \
0  0.282807  0.131713    0.03483    0.127129    0.31891          52.720674   

   median_inference_ms        FPS  
0            52.575821  18.967891  
        class   AP50_95
0      person  0.166754
1     bicycle  0.073378
2         car  0.283742
3  motorcycle  0.016271
4         bus  0.163120
5       truck  0.155192
Saved to: /content/drive/MyDrive/Dissertation/Results/bddk_crossdataset/Faster_rcnn/DAWN_BDDK
